In [0]:
df = spark.table("healthcare_analytics.bronze.claims_raw")

In [0]:
print(f"Total records: {df.count():,}")
df.printSchema()

In [0]:
# Explore payer type distribution
df.groupBy("payer_type").count().orderBy("count", ascending=False).show()

# Verify distribution across states
df.groupBy("patient_state").count().orderBy("count", ascending=False).show(10)

In [0]:
# Check patient gender distribution
df.groupBy("patient_gender").count().show()

In [0]:
# Identify most common diagnosis codes
df.groupBy("diagnosis_code").count().orderBy("count", ascending=False).show(10)

In [0]:
from pyspark.sql.functions import col

# Assess data quality for key business fields
key_columns = [
    "claim_id",
    "hvid",
    "prov_rendering_npi",
    "payer_type",
    "diagnosis_code",
    "procedure_code"
]
for col_name in key_columns:
    print(f"{col_name}: {df.filter(col(col_name).isNull()).count():,}")

In [0]:
%sql

-- Summarize key dataset characteristics
SELECT 
    COUNT(*) AS total_claims,
    COUNT(DISTINCT hvid) AS unique_patients,
    COUNT(DISTINCT prov_rendering_npi) as unique_providers,
    COUNT(DISTINCT diagnosis_code) as unique_diagnoses,
    COUNT(DISTINCT payer_type) AS unique_payer_types
FROM healthcare_analytics.bronze.claims_raw;

In [0]:
%sql

-- Verify each patient has consistent demographic attributes
SELECT
    hvid,
    COUNT(DISTINCT patient_gender) AS distinct_genders,
    COUNT(DISTINCT patient_year_of_birth) AS distinct_birth_years,
    COUNT(DISTINCT patient_state) AS distinct_states,
    COUNT(DISTINCT patient_zip3) AS distinct_zip3s
FROM healthcare_analytics.bronze.claims_raw
GROUP BY hvid
HAVING distinct_genders > 1 OR distinct_birth_years > 1 OR distinct_states > 1 OR distinct_zip3s > 1;



In [0]:
%sql

-- Compare total claim records against unique patients
SELECT
    COUNT(*) AS total_claims,
    COUNT(DISTINCT hvid) as unique_patients
FROM healthcare_analytics.bronze.claims_raw
WHERE hvid IS NOT NULL;

In [0]:
# Determine the numver of unique diagnosis, procedure codes, and provider taxonomies
procedure_df = spark.table("healthcare_analytics.bronze_silver.stg_procedures")
diagnosis_df = spark.table("healthcare_analytics.bronze_silver.stg_diagnoses")
provider_df = spark.table("healthcare_analytics.bronze_silver.stg_providers")

print(f"Unique procedure codes: {procedure_df.select('procedure_code').distinct().count():,}")
print(f"Unique diagnosis codes: {diagnosis_df.select('diagnosis_code').distinct().count():,}")
print(f"Unique provider taxonomies: {provider_df.select('provider_taxonomy').distinct().count():,}")

In [0]:
%sql

-- Validate one record per patient in the patient dimension
SELECT COUNT(*) as patient_records, COUNT(DISTINCT patient_id) as unique_patients
FROM healthcare_analytics.bronze_gold.dim_patients;

In [0]:
%sql

-- Validate one record per payer in the payer dimension
SELECT
    COUNT(*) AS payer_records,
    COUNT(DISTINCT payer_id) AS unique_payers
FROM healthcare_analytics.bronze_gold.dim_payers;

In [0]:
%sql

-- Validate provider dimension record counts
SELECT COUNT(*) AS provider_records, 
COUNT(DISTINCT provider_id) AS unique_providers, 
COUNT(DISTINCT provider_taxonomy) AS unique_provider_taxonomies 
FROM healthcare_analytics.bronze_gold.dim_providers;